In [ ]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 6.8 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.model_selection import train_test_split
from keras.models import Sequential
from keras.layers import Dense, Dropout, Activation, Embedding, BatchNormalization, Multiply
from sklearn.metrics import (precision_score, recall_score,f1_score, accuracy_score,mean_squared_error,mean_absolute_error)
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.preprocessing import Normalizer
import optuna
import tensorflow as tf
from keras.optimizers import Adam
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint
import json
import os
import pickle

In [ ]:
import logging

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(message)s'
)

In [ ]:
RAND = 1337
np.random.seed(RAND)

In [ ]:
DATA_DIR = os.path.dirname(os.path.abspath(__file__)) if "__file__"in locals() else "./"

In [ ]:
train = pd.read_csv(os.path.join(DATA_DIR, 'Training.csv'), header=None)
test = pd.read_csv(os.path.join(DATA_DIR, 'Testing.csv'), header=None)

In [ ]:
logging.info("Data Preprocessing...")

In [ ]:
X_train_df = train.iloc[:, 1:42]
y_train_df = train.iloc[:, 0]
X_test_df = test.iloc[:, 1:42]
y_test_df = test.iloc[:, 0]

X_train_df, X_val_df, y_train_df, y_val_df = train_test_split(
    X_train_df, y_train_df, test_size=0.2, random_state=RAND
)

scaler = Normalizer().fit(X_train_df)

X_train = np.array(scaler.transform(X_train_df))
X_val = np.array(scaler.transform(X_val_df))
X_test = np.array(scaler.transform(X_test_df))

y_train = np.array(y_train_df)
y_val = np.array(y_val_df)
y_test = np.array(y_test_df)

In [ ]:
print("Train Infs:", np.isinf(X_train).sum())
print("Val Infs:", np.isinf(X_val).sum())
print("Test Infs:",  np.isinf(X_test).sum())
print("Max:", X_train.max(), "Min:", X_train.min())
print("Label values:", np.unique(y_train))

Train Infs: 0
Val Infs: 0
Test Infs: 0
Max: 0.9999999999999263 Min: 0.0
Label values: [0 1]


In [ ]:
logging.info(f"X_train shape: {X_train.shape}")

In [ ]:
BATCH_SIZE=64
EPOCHS=1000

In [ ]:
logging.info("Starting hyperparameter optimization")

In [ ]:
strategy = tf.distribute.MirroredStrategy()

def objective(trial):
  with strategy.scope():
    inputs = tf.keras.Input(shape=(X_train.shape[1],))

    x = Dense(1024, activation='relu')(inputs)
    x = BatchNormalization()(x)
    x = Dropout(0.2)(x)

    x = Dense(768, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.2)(x)

    x = Dense(512, activation='relu')(x)
    x = Dropout(0.2)(x)

    # Camada de atenção
    attention = Dense(512, activation='softmax')(x)
    x = Multiply()([x, attention])

    output = Dense(1, activation='sigmoid')(x)

    model = tf.keras.Model(inputs=inputs, outputs=output)

    lr = trial.suggest_float("lr", 1e-5, 1e-1, log=True)

    model.compile(loss='binary_crossentropy',optimizer=Adam(learning_rate=lr),metrics=['accuracy'])

  early_stop = EarlyStopping(
      monitor='val_loss',
      patience=10,
      restore_best_weights=True
  )


  model.fit(
    X_train, y_train,
    batch_size=BATCH_SIZE,
    epochs=100,
    validation_data=(X_val, y_val),
    callbacks=[early_stop],
    verbose=0
  )


  loss, accuracy = model.evaluate(X_val, y_val, verbose=0)

  return accuracy

In [ ]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)

[I 2026-06-03 00:58:11,614] A new study created in memory with name: no-name-dc0492e8-a62a-40e2-8b06-27d0a1b052f0
[I 2026-06-03 01:40:03,368] Trial 0 finished with value: 0.9992409348487854 and parameters: {'lr': 0.00014738007181439803}. Best is trial 0 with value: 0.9992409348487854.


In [ ]:
logging.info("Best hyperparameters:")
for key, value in study.best_params.items():
  logging.info(f"{key}: {value}")

In [ ]:
best_lr = study.best_params['lr']

In [ ]:
checkpoint = ModelCheckpoint(
    os.path.join(DATA_DIR, 'checkpoints', 'checkpoint.keras'),
    monitor='val_loss',
    save_best_only=True
)

In [ ]:
from keras.losses import BinaryCrossentropy
from keras.optimizers import Adam

In [ ]:
with strategy.scope():
  inputs = tf.keras.Input(shape=(X_train.shape[1],))

  x = Dense(1024, activation='relu')(inputs)
  x = BatchNormalization()(x)
  x = Dropout(0.2)(x)

  x = Dense(768, activation='relu')(x)
  x = BatchNormalization()(x)
  x = Dropout(0.2)(x)

  x = Dense(512, activation='relu')(x)
  x = Dropout(0.2)(x)

  # Camada de atenção
  attention = Dense(512, activation='softmax')(x)
  x = Multiply()([x, attention])

  output = Dense(1, activation='sigmoid')(x)

  model = tf.keras.Model(inputs=inputs, outputs=output)

  model.summary()

  model.compile(loss='binary_crossentropy',optimizer=Adam(learning_rate=best_lr),metrics=['accuracy'])


In [ ]:
early_stop = EarlyStopping(
    monitor='val_loss',
    patience=10,
    restore_best_weights=True
)

In [ ]:
model.fit(
  X_train, y_train,
  batch_size=BATCH_SIZE,
  epochs=EPOCHS,
  validation_data=(X_val, y_val),
  callbacks=[early_stop, checkpoint],
  verbose=1
)
model.save('./models/model_new_kdd_v1.keras')

In [ ]:
y_pred = (model.predict(X_test) > 0.5).astype(int).flatten()

In [ ]:
accuracy = accuracy_score(y_test, y_pred)
recall = recall_score(y_test, y_pred , average="binary")
precision = precision_score(y_test, y_pred , average="binary")
f1 = f1_score(y_test, y_pred, average="binary")
print("----------------------------------------------")
print("accuracy")
print("%.3f" %accuracy)
print("recall")
print("%.3f" %recall)
print("precision")
print("%.3f" %precision)
print("f1score")
print("%.3f" %f1)